In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from scipy.stats import norm
import os

from model import *
from data import generate_gmm_data, generate_random_data
from train_utils import train

from plot import *
from utils import *


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
relu = lambda X: np.maximum(X, 0)
d_relu = lambda X: np.maximum(np.sign(X), 0)

elu = lambda X, alpha=1.0: ( alpha * (np.exp(X) - 1) ) * (X <= 0) + (X * (X > 0))
d_elu = lambda X, alpha=1.0: ( alpha * np.exp(X) ) * (X <= 0) + (X > 0)

gelu = lambda X: X * norm.cdf(X)
d_gelu = lambda X: norm.cdf(X) + X * norm.pdf(X)

sigmoid = lambda X: 1 / (1 + np.exp(-X))
d_sigmoid = lambda X: sigmoid(X) * (1 - sigmoid(X))

silu = lambda X: X * sigmoid(X)
d_silu = lambda X: sigmoid(X) + X * d_sigmoid(X)

leaky_relu = lambda X, slope=0.01: slope * np.minimum(X, 0) + np.maximum(X, 0)
d_leaky_relu = lambda X, slope=0.01: slope * (X <= 0) + (X > 0).astype(float)

identity = lambda X: X
d_identity = lambda X: np.ones(X.shape)

abs_ = lambda X: np.abs(X)
d_abs_ = lambda X: -1 * (X <= 0) + (X > 0)

softplus = lambda X: np.log(1 + np.exp(X))
d_softplus = lambda X: sigmoid(X)

leaky_softplus = lambda X, slope=0.01: slope * X + (1 - slope) * (softplus(X) - np.log(2))
d_leaky_softplus = lambda X, slope=0.01: slope + (1 - slope) * (d_softplus(X))


smooth_abs = lambda X, beta = 10: (2 / beta) * softplus(X) - X - ( (2 / beta) * np.log(2) )
d_smooth_abs = lambda X, beta = 10: (2 / beta) * d_softplus(X) - 1

custom_activations_map = {'ReLU': [relu, d_relu],
               'Leaky-Softplus': [leaky_softplus, d_leaky_softplus], 
               'ELU': [elu, d_elu],
               'GELU': [gelu, d_gelu],
               'SiLU': [silu, d_silu],
               'Leaky-ReLU': [leaky_relu, d_leaky_relu],
               'Abs': [abs_, d_abs_],
               'SmoothAbs': [smooth_abs, d_smooth_abs],
               'Identity': [identity, d_identity]}

In [ ]:
optimizer_map = {'SGD': optim.SGD,
                'Adam': optim.Adam}

# Model Training

In [ ]:
def get_data(d: int, 
             K: int, 
             N_k: int, 
             nu: float = 1.0,
             normalize_: bool = False, 
             orthogonalize_: bool = False):
    
    
    N = K * N_k
    
    # d > N --> random Gaussian data
    if d > N:
        train_set, train_loader, _ = generate_random_data(d, 
                                                       K, 
                                                       N_k, 
                                                       orthogonalize_ = orthogonalize_, 
                                                       normalize_ = normalize_)
        val_loader = train_loader 
    
    # N > d --> Gaussian mixture data
    elif N > d:
        train_set, train_loader, _ = generate_gmm_data(d, 
                                                    K, 
                                                    N_k, 
                                                    whiten_ = whiten_, 
                                                    normalize_ = normalize_)
        val_loader = train_loader
    
    
    X = train_set.samples.T
    Y = train_set.targets.T
    
    return train_loader, val_loader, X, Y

In [ ]:
def train_model(num_layers: int,
                data_dim: int, 
                hidden_dim: int, 
                num_classes: int, 
                act_fn_str: str, 
                train_loader, 
                val_loader, 
                epochs, 
                lr, 
                freq,
                loss_fn, 
                optimizer_type='SGD', 
                freeze_last_layer=True, 
                residual=False, 
                batchnorm=False, 
                init_method='orthogonal',
                init_scale=1):
    
    
    # Train model
    model = MLP(num_layers, 
                data_dim, 
                hidden_dim, 
                num_classes, 
                act_fn_str, 
                freeze_last_layer=freeze_last_layer,
                residual=residual, 
                batchnorm=batchnorm, 
                init_method=init_method, 
                init_scale=init_scale)

    model = model.to(device)

    # Training
    criterion = loss_map[loss_fn]
    optimizer = optimizer_map[optimizer_type](model.parameters(), lr=lr, weight_decay=0)
    scheduler = None
    #scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs, eta_min=0)
    results = train(model, 
                    criterion, 
                    optimizer, 
                    train_loader, 
                    val_loader, 
                    epochs, 
                    device,
                    scheduler=scheduler, 
                    return_weights=True,
                    return_grads=True,
                    return_outs=False)

    grads = results['all_grads']
    train_losses = results['train_losses']
    W2 = results['all_weights'][0][1]
    #outs = results['all_outs']
    
    return grads, W2, train_losses

# Top singular values

In [ ]:
'''
Helper function to calculate the norm of each component in the weight matrix 
'''
def grad_norm_and_svals_ratio(grads, K, init_scale, epochs):
    
    max_top_sval_ratio = np.zeros((epochs))
    grad_norm_ratio = np.zeros((epochs))
    max_tail_svals = np.zeros((epochs))
    
    G1_0 = grads[0][0]
    s0_top = np.linalg.svd(G1_0, compute_uv=False)[:K]
    
    for t in range(epochs):
        G1_t = grads[t][0]
        
        # Gradient norm
        grad_norm_ratio[t] = np.linalg.norm(G1_t, ord='fro') / np.linalg.norm(G1_0, ord='fro')
        
        # Singular values
        st_top = np.linalg.svd(G1_t, compute_uv=False)[:(K + 1)]
        
        sval_ratio = st_top[:-1] / s0_top
        max_top_sval_ratio[t] = np.max(sval_ratio)
        
        max_tail_svals[t] = st_top[-1]
        
        
    return grad_norm_ratio, max_top_sval_ratio, max_tail_svals 

In [ ]:
reduction = 'sum'
loss_map = {
    'ce_loss': nn.CrossEntropyLoss(reduction=reduction),
    'mse_loss': nn.MSELoss(reduction=reduction)
}

data_type_map = {
    'gmm': 'GMM', 
    'uos': 'UoS',
    'random': 'Random'
}

In [ ]:
# Data setup
d = 512 # Ambient dim
m = 768
K = 4 # Number of classes
N_k = 25
N = N_k * K
nu = 1

# Model setup
L = 2
act_fn_str = 'Leaky-Softplus' 
residual = False
batchnorm = False
freeze_last_layer = True
init_method = 'orthogonal'
init_scale = 1e-2

whiten_ = False
normalize_ = False
orthogonalize_ = False

# Training setup
epochs = 1500
lrs = [1e-3, 1e-4, 1e-5]
freq = 1
optimizer_type = 'SGD'
#loss_fn = 'ce_loss'
loss_fn = 'mse_loss'

savefig_dir = './figs/assumptions/'
if not os.path.exists(savefig_dir):
    os.makedirs(savefig_dir)


In [ ]:
trials = 5


grad_norm_ratio = np.zeros((trials, len(lrs), epochs))
max_top_sval_ratio = np.zeros((trials, len(lrs), epochs))


train_losses = np.zeros((trials, len(lrs), epochs))

for j, lr in enumerate(lrs):
    print("====== (Init scale, Learning rate) = (", init_scale, ",", lr, ") ======")

    for i in range(trials):
        # Create data
        train_loader, val_loader, X, Y = get_data(d, K, N_k, nu, whiten_, normalize_)

        print("=== TRIAL " + str(i) + " ===")

        # Train model
        grads, W2, train_losses_trial = train_model(L, d, m, K, act_fn_str, 
                                                         train_loader, 
                                                         val_loader, 
                                                         epochs, 
                                                         lr, 
                                                         freq, 
                                                         loss_fn, 
                                                         optimizer_type=optimizer_type,
                                                         freeze_last_layer=freeze_last_layer, 
                                                         residual=residual,
                                                         batchnorm=batchnorm,
                                                         init_method=init_method,
                                                         init_scale=init_scale)


        # Plot train loss
        plot_train_losses(train_losses_trial, act_fn_str, d, d, K, None, savefig=False)
        train_losses[i, j, :] = train_losses_trial

        # Calculate norms of decompositions
        trial_grad_norm_ratio, trial_max_top_sval_ratio, _ = grad_norm_and_svals_ratio(grads, K, init_scale, epochs)

        grad_norm_ratio[i, j :] = trial_grad_norm_ratio
        max_top_sval_ratio[i, j, :] = trial_max_top_sval_ratio            



In [ ]:
np.savez(os.path.join(savefig_dir, 'grad_norm_ratio.npz'), grad_norm_ratio=grad_norm_ratio)
np.savez(os.path.join(savefig_dir, 'max_top_sval_ratio.npz'), max_top_sval_ratio=max_top_sval_ratio)

In [ ]:
# Mean and stdev of weight decomposition norms across trials
mean_grad_norm_ratio = np.mean(grad_norm_ratio, axis=0)
std_grad_norm_ratio = np.std(grad_norm_ratio, axis=0)

mean_max_top_sval_ratio = np.mean(max_top_sval_ratio, axis=0)
std_max_top_sval_ratio = np.std(max_top_sval_ratio, axis=0)


# Plotting
x_range = np.arange(epochs)

for j, lr in enumerate(lrs):
    plt.figure(figsize=(8, 6))
    # Plot gradient norm ratio from initialization
    plt.plot(x_range, 
               mean_grad_norm_ratio[j, :], 
               linewidth=3,
               label=r'$\frac{\| G_1(t) \|_F}{\| G_1(0) \|_F}$')

    plt.fill_between(x_range, 
                     mean_grad_norm_ratio[j, :] - std_grad_norm_ratio[j, :], 
                     mean_grad_norm_ratio[j, :] + std_grad_norm_ratio[j, :],
                     linewidth=3,
                     alpha=0.5)


    # Plot max top singular value ratio
    plt.plot(x_range, 
               mean_max_top_sval_ratio[j, :], 
               linewidth=3,
               label=r'$\max_{i \in [K]} \; \frac{\sigma_i(G_1(t))}{\sigma_i(G_1(0))}$')

    plt.fill_between(x_range, 
                     mean_max_top_sval_ratio[j, :] - std_max_top_sval_ratio[j, :], 
                     mean_max_top_sval_ratio[j, :] + std_max_top_sval_ratio[j, :],
                     linewidth=3,
                     alpha=0.5)
    
    plt.plot(x_range, 
               2 * mean_grad_norm_ratio[j, :], 
               linewidth=3,
               label=r'$G_U \cdot \frac{\| G_1(t) \|_F}{\| G_1(0) \|_F}$')

    plt.fill_between(x_range, 
                     2 * (mean_grad_norm_ratio[j, :] - std_grad_norm_ratio[j, :]), 
                     2 * (mean_grad_norm_ratio[j, :] + std_grad_norm_ratio[j, :]),
                     linewidth=3,
                     alpha=0.5)
        
        
       
        

    # Show figure 
    title = r'$\eta = 10^{' + str( int( np.log10(lr) ) ) + r'}, \epsilon = 10^{' + str( int(np.log10(init_scale)) ) + r'}$'
    plt.title(title, fontsize=24, y=1.2)
    plt.xlabel('Train Epoch', fontsize=20)
    plt.ylabel(r'Ratio', 
                fontsize=20,
                 x = 0.01)

    plt.grid('on')
    plt.xticks(fontsize=16) 
    plt.yticks(fontsize=16)
    plt.yscale('log')


    plt.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, 1.2), 
        ncol=3,
        fontsize=16,
        frameon=False
    )

    plt.tight_layout()
    save_fname = 'assumption_grad_norm_top_sval_ratio_lr_' + str(lr) + '.pdf'

    plt.savefig(os.path.join(savefig_dir, save_fname), bbox_inches='tight')
    plt.show()
    plt.close()

# Tail singular values

In [ ]:
# Data setup
d = 512 # Ambient dim
m = 768
K = 4 # Number of classes
N_k = 25
N = N_k * K
nu = 1

# Model setup
L = 2
act_fn_str = 'Leaky-Softplus' 
residual = False
batchnorm = False
freeze_last_layer = True
init_method = 'orthogonal'
init_scales = [1e-2, 1e-3, 1e-4, 1e-5]

whiten_ = False
normalize_ = False
orthogonalize_ = False

# Training setup
epochs = 20
lrs = [1e-3, 1e-4, 1e-5]
freq = 1
optimizer_type = 'SGD'
#loss_fn = 'ce_loss'
loss_fn = 'mse_loss'

savefig_dir = './figs/assumptions/'
if not os.path.exists(savefig_dir):
    os.makedirs(savefig_dir)


In [ ]:
trials = 1


grad_norm_ratio = np.zeros((trials, len(lrs), len(init_scales), epochs))
max_top_sval_ratio = np.zeros((trials, len(lrs), len(init_scales), epochs))

max_tail_sval = np.zeros((trials, len(lrs), len(init_scales), epochs))
W2YX_Kth_sval = np.zeros((trials, len(lrs), len(init_scales)))


train_losses = np.zeros((trials, len(lrs), len(init_scales), epochs))

for j, lr in enumerate(lrs):
    for k, init_scale in enumerate(init_scales):
        print("====== (Init scale, Learning rate) = (", init_scale, ",", lr, ") ======")
        
        for i in range(trials):
            # Create data
            train_loader, val_loader, X, Y = get_data(d, K, N_k, nu, whiten_, normalize_)

            print("=== TRIAL " + str(i) + " ===")

            # Train model
            grads, W2, train_losses_trial = train_model(L, d, m, K, act_fn_str, 
                                                             train_loader, 
                                                             val_loader, 
                                                             epochs, 
                                                             lr, 
                                                             freq, 
                                                             loss_fn, 
                                                             optimizer_type=optimizer_type,
                                                             freeze_last_layer=freeze_last_layer, 
                                                             residual=residual,
                                                             batchnorm=batchnorm,
                                                             init_method=init_method,
                                                             init_scale=init_scale)


            # Plot train loss
            plot_train_losses(train_losses_trial, act_fn_str, d, d, K, None, savefig=False)
            train_losses[i, j, k, :] = train_losses_trial

            # Get max residual
            #max_resids[i, ...] = np.array( [ np.max(np.abs( out.T - Y )) for out in outs] )

            # Calculate norms of decompositions
            trial_grad_norm_ratio, trial_max_top_sval_ratio, trial_max_tail_sval = grad_norm_and_svals_ratio(grads, K, init_scale, epochs)
            
            
            grad_norm_ratio[i, j, k, :] = trial_grad_norm_ratio
            max_top_sval_ratio[i, j, k, :] = trial_max_top_sval_ratio
            
            max_tail_sval[i, j, k, :] = trial_max_tail_sval
            W2YX_Kth_sval[i, j, k] = np.linalg.svd(W2.T @ Y @ X.T, compute_uv=False)[K - 1]
            
            


In [ ]:
np.savez(os.path.join(savefig_dir, 'max_tail_svals.npz'), max_tail_sval=max_tail_sval)
np.savez(os.path.join(savefig_dir, 'W2YX_Kth_sval.npz'), W2YX_Kth_sval=W2YX_Kth_sval)


In [ ]:
# Mean and stdev of weight decomposition norms across trials
mean_max_tail_sval = np.mean(max_tail_sval, axis=0)
std_max_tail_sval = np.std(max_tail_sval, axis=0)

mean_W2YX_Kth_sval = np.mean(W2YX_Kth_sval, axis=0)
std_W2YX_Kth_sval = np.std(W2YX_Kth_sval, axis=0)


# Plotting
x_range = np.arange(epochs)

for j, lr in enumerate(lrs):
    plt.figure(figsize=(8, 6))
    for k, init_scale in enumerate(init_scales):
        # Plot gradient norm ratio from initialization
        plt.plot(x_range, 
                   mean_max_tail_sval[j, k, :], 
                   linewidth=3,
                   label=r'$\epsilon = 10^{' + str( int(np.log10(init_scale)) ) + r'}$')

        plt.fill_between(x_range, 
                         mean_max_tail_sval[j, k, :] - std_max_tail_sval[j, k, :], 
                         mean_max_tail_sval[j, k, :] + std_max_tail_sval[j, k, :],
                         linewidth=3,
                         alpha=0.5)
        
    plt.plot(x_range, 
            mean_W2YX_Kth_sval[j, k] * np.ones(len(x_range)),
            linewidth=3, 
            linestyle='dashed', color='black') 
            #label=r'$\sigma_K(W_2^\top YX^\top)$')
    
    plt.fill_between(x_range,
                    mean_W2YX_Kth_sval[j, k] * np.ones(len(x_range)) - std_W2YX_Kth_sval[j, k] * np.ones(len(x_range)), 
                    mean_W2YX_Kth_sval[j, k] * np.ones(len(x_range)) + std_W2YX_Kth_sval[j, k] * np.ones(len(x_range)), 
                    linewidth=3, 
                    alpha=0.5, linestyle='dashed', color='black')


    # Show figure 
    title = r'$\eta = 10^{' + str( int( np.log10(lr) ) ) + r'}$'
    plt.title(title, fontsize=24, y=1.2)
    plt.xlabel('Train Epoch', fontsize=20)
    plt.ylabel(r'$\sigma_{K + 1}(G_1(t))$', 
                  fontsize=20,
                 x = 0.01)

    plt.grid('on')
    plt.xticks(fontsize=16) 
    plt.yticks(fontsize=16)
    plt.yscale('log')


    plt.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, 1.2), 
        ncol=len(init_scales),
        fontsize=16,
        frameon=False
    )

    plt.tight_layout()
    save_fname = 'assumption_grad_tail_svals_lr_' + str(lr) + '.pdf'

    plt.savefig(os.path.join(savefig_dir, save_fname), bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# # Mean and stdev of weight decomposition norms across trials
# mean_max_resids = np.mean(max_resids, axis=0)
# std_max_resids = np.std(max_resids, axis=0)



# # Plotting
# x_range = np.arange(epochs)
# plt.figure(figsize=(8, 5))


# # Plot gradient norm ratio from initialization
# plt.plot(x_range, 
#            mean_max_resids, 
#            linewidth=3)

# plt.fill_between(x_range, 
#                  mean_max_resids - std_max_resids, 
#                  mean_max_resids + std_max_resids,
#                  linewidth=3,
#                  alpha=0.5)


# # Plot singular value maximum ratio
# # plt.plot(x_range, 
# #          mean_max_sval_ratios, 
# #          linewidth=3,
# #          label=r'$\max_i \: \frac{\sigma_i(G_1(t))}{\sigma_i(G_1(0))}$')

# # plt.fill_between(x_range, 
# #                  mean_max_sval_ratios - std_max_sval_ratios, 
# #                  mean_max_sval_ratios + std_max_sval_ratios,
# #                  linewidth=3,
# #                  alpha=0.5)


# # plt.plot(x_range, 
# #          mean_min_sval_ratios, 
# #          linewidth=3,
# #          label=r'$\min_i \frac{\sigma_i(G_1(t))}{\sigma_i(G_1(0))}$')

# # plt.fill_between(x_range, 
# #                  mean_min_sval_ratios - std_min_sval_ratios, 
# #                  mean_min_sval_ratios + std_min_sval_ratios,
# #                  linewidth=3,
# #                  alpha=0.5)



# # Show figure 
# #title = r'$d > N$' if d > N else r'$N > d$'
# #plt.title(title, fontsize=24, y=1.15)
# plt.xlabel('Train Epoch', fontsize=24)
# plt.ylabel('Max Residual', 
#               fontsize=24,
#              x = 0.01)

# plt.grid('on')
# plt.xticks(fontsize=16) 
# plt.yticks(fontsize=16)


# # plt.legend(
# #     loc='upper center',
# #     bbox_to_anchor=(0.5, 1.225), 
# #     ncol=4,
# #     fontsize=16,
# #     frameon=False
# # )

# savefig_dir = './figs/assumptions/'
# if not os.path.exists(savefig_dir):
#     os.makedirs(savefig_dir)

# plt.tight_layout()
# save_fname = 'assumption_resids.pdf'

# #plt.savefig(os.path.join(savefig_dir, save_fname), bbox_inches='tight')
# plt.show()
# plt.close()

In [ ]:
# print(max_resids)